# Function Calling in OpenAI

In [19]:
import os
import openai

In [20]:
from getpass import getpass
# api_key=getpass("OpenAI API Key: ")
openai.api_key= getpass("OpenAI API Key: ")

In [21]:
prompt = "Hello, How are you?"

response = openai.chat.completions.create(
    model = "gpt-3.5-turbo",
    messages = [
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": prompt}
    ]
)

print(response.choices[0].message.content)

Hello! I'm here and ready to help. How can I assist you today?


https://rapidapi.com/hub

In [22]:
# Example dummy function hard coded to return the same weather
# In production, this could be your backend API or an external API

import requests
def get_current_weather(location):
    """Get the current weather in a given location"""

    url = "https://ai-weather-by-meteosource.p.rapidapi.com/find_places"

    querystring = {"text":location}

    headers = {
      'x-rapidapi-key': "156877d1e1msh6806b57dfec44b9p1bd2c8jsn796f2ff15790",
      'x-rapidapi-host': "ai-weather-by-meteosource.p.rapidapi.com"
    }

    response = requests.get(url, headers=headers, params=querystring)

    print(response.json())
  
    return response.json()


In [23]:
response = get_current_weather("Delhi")

[{'name': 'Delhi', 'place_id': 'delhi', 'adm_area1': 'National Capital Territory of Delhi', 'adm_area2': None, 'country': 'India', 'lat': '28.65195N', 'lon': '77.23149E', 'timezone': 'Asia/Kolkata', 'type': 'settlement'}, {'name': 'Dili', 'place_id': 'dili', 'adm_area1': 'Díli', 'adm_area2': 'Vera Cruz', 'country': 'Timor-Leste', 'lat': '8.55861S', 'lon': '125.57361E', 'timezone': 'Asia/Dili', 'type': 'settlement'}, {'name': 'Delhi', 'place_id': 'delhi-5342522', 'adm_area1': 'California', 'adm_area2': 'Merced', 'country': 'United States of America', 'lat': '37.43216N', 'lon': '120.77854W', 'timezone': 'America/Los_Angeles', 'type': 'settlement'}, {'name': 'Delhi', 'place_id': 'delhi-5114824', 'adm_area1': 'New York', 'adm_area2': 'Delaware', 'country': 'United States of America', 'lat': '42.27814N', 'lon': '74.91599W', 'timezone': 'America/New_York', 'type': 'settlement'}, {'name': 'Delhi', 'place_id': 'delhi-4321929', 'adm_area1': 'Louisiana', 'adm_area2': 'Richland', 'country': 'Unit

In [24]:
response

[{'name': 'Delhi',
  'place_id': 'delhi',
  'adm_area1': 'National Capital Territory of Delhi',
  'adm_area2': None,
  'country': 'India',
  'lat': '28.65195N',
  'lon': '77.23149E',
  'timezone': 'Asia/Kolkata',
  'type': 'settlement'},
 {'name': 'Dili',
  'place_id': 'dili',
  'adm_area1': 'Díli',
  'adm_area2': 'Vera Cruz',
  'country': 'Timor-Leste',
  'lat': '8.55861S',
  'lon': '125.57361E',
  'timezone': 'Asia/Dili',
  'type': 'settlement'},
 {'name': 'Delhi',
  'place_id': 'delhi-5342522',
  'adm_area1': 'California',
  'adm_area2': 'Merced',
  'country': 'United States of America',
  'lat': '37.43216N',
  'lon': '120.77854W',
  'timezone': 'America/Los_Angeles',
  'type': 'settlement'},
 {'name': 'Delhi',
  'place_id': 'delhi-5114824',
  'adm_area1': 'New York',
  'adm_area2': 'Delaware',
  'country': 'United States of America',
  'lat': '42.27814N',
  'lon': '74.91599W',
  'timezone': 'America/New_York',
  'type': 'settlement'},
 {'name': 'Delhi',
  'place_id': 'delhi-4321929

In [25]:
functions = [
        {
            "name": "get_current_weather",
            "description": "Get the current weather in a given location",
            "parameters": {
                "type": "object",
                "properties": {
                    "location": {
                        "type": "string",
                        "description": "The city and state, e.g. San Francisco, CA",
                    },
                    
                },
                "required": ["location"],
            },
        }
    ]

In [26]:

user_message="Hi There"
messages=[]
messages.append({"role": "user", "content":user_message})

completion=openai.chat.completions.create(
    model="gpt-3.5-turbo",
    messages= messages
    
)

In [ ]:
print(completion.choices[0].message.content)

ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=[], audio=None, function_call=FunctionCall(arguments='{"location":"Bangalore"}', name='get_current_weather'), tool_calls=None)


In [28]:
messages

[{'role': 'user', 'content': 'Hi There'}]

In [29]:
user_message="What is the temperature of Bangalore"

messages.append({"role": "user", "content": user_message})

completion=openai.chat.completions.create(
    model="gpt-3.5-turbo",
    messages=messages,
    functions=functions
    
)

In [32]:
print(completion.choices[0].message)

ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=[], audio=None, function_call=FunctionCall(arguments='{"location":"Bangalore"}', name='get_current_weather'), tool_calls=None)


In [33]:
response=completion.choices[0].message

In [34]:
response

ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=[], audio=None, function_call=FunctionCall(arguments='{"location":"Bangalore"}', name='get_current_weather'), tool_calls=None)

In [38]:
function_name=response.function_call.name
print(function_name)


get_current_weather


In [42]:
response.function_call.arguments

'{"location":"Bangalore"}'

In [44]:
import json
location=eval(response.function_call.arguments)['location']
print(location)

Bangalore


In [45]:
# Step 4: send the info on the function call and function response to GPT
messages.append(response)  # extend conversation with assistant's reply
messages.append(
    {
        "role": "function",
        "name": function_name,
        "content": location,
    }
)

In [46]:
messages

[{'role': 'user', 'content': 'Hi There'},
 {'role': 'user', 'content': 'What is the temperature of Bangalore'},
 ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=[], audio=None, function_call=FunctionCall(arguments='{"location":"Bangalore"}', name='get_current_weather'), tool_calls=None),
 {'role': 'function', 'name': 'get_current_weather', 'content': 'Bangalore'}]

In [48]:
# extend conversation with function response
second_response = openai.chat.completions.create(
    model="gpt-3.5-turbo",
    messages=messages,
    functions=functions
)  # get a new response from GPT where it can see the function response


In [50]:
print(second_response.choices[0].message.content)

I am sorry, but I am unable to fetch the current weather information for Bangalore at the moment.
